In [ ]:
!pip -q install transformers datasets sentencepiece accelerate evaluate spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 93.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import json
import re
import torch
import spacy

from tqdm import tqdm
from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)

from google.colab import files

# Upload Mintaka files
uploaded = files.upload()

# Load spaCy model
nlp = spacy.load("en_core_web_sm")

device = "cuda" if torch.cuda.is_available() else "cpu"

print(device)

Saving mintaka_dev.json to mintaka_dev (1).json
Saving mintaka_test.json to mintaka_test (1).json
Saving mintaka_train.json to mintaka_train (1).json
cuda


In [ ]:
def extract_answer(item):

    ans = item.get("answer", {})

    if ans is None:
        return ""

    if isinstance(ans, dict):

        if "answer" in ans and ans["answer"] is not None and len(ans["answer"]) > 0:

            obj = ans["answer"][0]

            if isinstance(obj, dict):

                if "label" in obj:

                    label = obj["label"]

                    if isinstance(label, dict):

                        return str(label.get("en", ""))

                if "name" in obj:
                    return str(obj["name"])

            return str(obj)

        if "mention" in ans:
            return str(ans["mention"])

    return ""


def load_mintaka(path):

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    questions = []
    answers = []

    for item in data:

        questions.append(item["question"])
        answers.append(extract_answer(item))

    return questions, answers


train_q, train_a = load_mintaka("mintaka_train.json")
dev_q, dev_a = load_mintaka("mintaka_dev.json")
test_q, test_a = load_mintaka("mintaka_test.json")

print(len(train_q), len(dev_q), len(test_q))
print(train_q[:3])
print(train_a[:10])

14000 2000 4000
['What is the seventh tallest mountain in North America?', 'Which actor was the star of Titanic and was born in Los Angeles, California?', 'Which actor starred in Vanilla Sky and was married to Katie Holmes?']
['Mount Lucania', 'Leonardo DiCaprio', 'Tom Cruise', '1996', 'Ron DeSantis', 'Joe Biden', '8', 'False', 'Henry Cavill', '20']


In [ ]:
MODEL_NAME = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

model = model.to(device)

print("Model Loaded!")
print(model.config._name_or_path)


# ----------------------------
# Lightweight NED
# ----------------------------

def extract_entities(question):

    doc = nlp(question)

    entities = []

    for ent in doc.ents:

        if ent.text.strip() != "":
            entities.append(ent.text.strip())

    # Remove duplicates while preserving order
    entities = list(dict.fromkeys(entities))

    return entities


# Test
print(train_q[0])
print(extract_entities(train_q[0]))
print(train_q[1])
print(extract_entities(train_q[1]))

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Model Loaded!
google/flan-t5-base
What is the seventh tallest mountain in North America?
['seventh', 'North America']
Which actor was the star of Titanic and was born in Los Angeles, California?
['Titanic', 'Los Angeles', 'California']


In [ ]:
def make_prompt(question):

    entities = extract_entities(question)

    if len(entities) == 0:
        entity_text = "None"
    else:
        entity_text = ", ".join(entities)

    prompt = f"""Answer the question.

Question:
{question}

Detected Entities:
{entity_text}

Answer:
"""

    return prompt


# Test
print(make_prompt(train_q[0]))
print("="*80)
print(make_prompt(train_q[1]))

Answer the question.

Question:
What is the seventh tallest mountain in North America?

Detected Entities:
seventh, North America

Answer:

Answer the question.

Question:
Which actor was the star of Titanic and was born in Los Angeles, California?

Detected Entities:
Titanic, Los Angeles, California

Answer:



In [ ]:
from tqdm import tqdm

def create_inputs(questions, answers):

    inputs = []
    labels = []

    for q, a in tqdm(zip(questions, answers), total=len(questions)):

        inputs.append(make_prompt(q))
        labels.append(str(a))

    return inputs, labels


# Create prompts
train_inputs, train_labels = create_inputs(train_q, train_a)

dev_inputs, dev_labels = create_inputs(dev_q, dev_a)

test_inputs, test_labels = create_inputs(test_q, test_a)


print("Train:", len(train_inputs))
print("Dev:", len(dev_inputs))
print("Test:", len(test_inputs))

print("\nExample Prompt:\n")
print(train_inputs[0])

print("\nAnswer:")
print(train_labels[0])

100%|██████████| 4000/4000 [00:30<00:00, 131.18it/s]

Train: 14000
Dev: 2000
Test: 4000

Example Prompt:

Answer the question.

Question:
What is the seventh tallest mountain in North America?

Detected Entities:
seventh, North America

Answer:


Answer:
Mount Lucania


In [ ]:
from datasets import Dataset

train_ds = Dataset.from_dict({
    "input_text": train_inputs,
    "target_text": train_labels
})

dev_ds = Dataset.from_dict({
    "input_text": dev_inputs,
    "target_text": dev_labels
})

test_ds = Dataset.from_dict({
    "input_text": test_inputs,
    "target_text": test_labels
})

print(train_ds)
print(dev_ds)
print(test_ds)

print("\nExample:")
print(train_ds[0])

Dataset({
    features: ['input_text', 'target_text'],
    num_rows: 14000
})
Dataset({
    features: ['input_text', 'target_text'],
    num_rows: 2000
})
Dataset({
    features: ['input_text', 'target_text'],
    num_rows: 4000
})

Example:
{'input_text': 'Answer the question.\n\nQuestion:\nWhat is the seventh tallest mountain in North America?\n\nDetected Entities:\nseventh, North America\n\nAnswer:\n', 'target_text': 'Mount Lucania'}


In [ ]:
def preprocess(batch):

    model_inputs = tokenizer(
        batch["input_text"],
        max_length=512,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        text_target=batch["target_text"],
        max_length=32,
        truncation=True,
        padding="max_length"
    )

    # Replace padding tokens with -100 so they are ignored in the loss
    label_ids = labels["input_ids"]

    label_ids = [
        [(token if token != tokenizer.pad_token_id else -100) for token in seq]
        for seq in label_ids
    ]

    model_inputs["labels"] = label_ids

    return model_inputs


train_ds = train_ds.map(
    preprocess,
    batched=True,
    remove_columns=train_ds.column_names
)

dev_ds = dev_ds.map(
    preprocess,
    batched=True,
    remove_columns=dev_ds.column_names
)

test_ds = test_ds.map(
    preprocess,
    batched=True,
    remove_columns=test_ds.column_names
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

print(train_ds[0])

Map:   0%|          | 0/14000 [00:00<?, ? examples/s]

KeyError: 'input_text'

In [ ]:
training_args = Seq2SeqTrainingArguments(

    output_dir="./flan_t5_lightweight_ned",

    learning_rate=3e-5,

    per_device_train_batch_size=4,

    per_device_eval_batch_size=4,

    num_train_epochs=3,

    weight_decay=0.01,

    predict_with_generate=True,

    eval_strategy="epoch",

    save_strategy="epoch",

    logging_steps=100,

    save_total_limit=2,

    fp16=False,

    gradient_accumulation_steps=1,

    max_grad_norm=1.0,

    report_to=[],

    load_best_model_at_end=False
)

In [ ]:
trainer = Seq2SeqTrainer(

    model=model,

    args=training_args,

    train_dataset=train_ds,

    eval_dataset=dev_ds,

    processing_class=tokenizer,

    data_collator=data_collator
)

In [ ]:
print(train_ds[0]["labels"][:20])

[7964, 3, 11748, 11219, 1, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100]


In [ ]:
print(train_ds.features)

{'input_ids': List(Value('int32')), 'attention_mask': List(Value('int8')), 'labels': List(Value('int64'))}


In [ ]:
print(train_ds[0])

{'input_ids': [11801, 8, 822, 5, 11860, 10, 363, 19, 8, 17353, 5065, 222, 4180, 16, 1117, 1371, 58, 374, 17, 7633, 4443, 2197, 10, 17353, 6, 1117, 1371, 11801, 10, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [ ]:
batch = data_collator([train_ds[0], train_ds[1]])

for k, v in batch.items():
    print(k, v.shape)

print(batch["labels"])

input_ids torch.Size([2, 512])
attention_mask torch.Size([2, 512])
labels torch.Size([2, 32])
decoder_input_ids torch.Size([2, 32])
tensor([[ 7964,     3, 11748, 11219,     1,  -100,  -100,  -100,  -100,  -100,
          -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
          -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
          -100,  -100],
        [17342,    32,  2043,   254,     9,  2246,    32,     1,  -100,  -100,
          -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
          -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
          -100,  -100]])


In [ ]:
model.eval()

batch = data_collator([train_ds[0], train_ds[1]])

batch = {k: v.to(device) for k, v in batch.items()}

with torch.no_grad():
    outputs = model(**batch)

print(outputs.loss)

tensor(nan, device='cuda:0')


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tok = AutoTokenizer.from_pretrained("google/flan-t5-base")
mdl = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base").to(device)

batch = data_collator([train_ds[0], train_ds[1]])
batch = {k: v.to(device) for k, v in batch.items()}

with torch.no_grad():
    out = mdl(**batch)

print(out.loss)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


tensor(3.1907, device='cuda:0')


In [ ]:
trainer.train()

trainer.save_model("./flan_t5_lightweight_ned")

tokenizer.save_pretrained("./flan_t5_lightweight_ned")

Epoch,Training Loss,Validation Loss
1,1.985972,1.647113
2,1.780517,1.612138
3,1.871057,1.601407


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./flan_t5_lightweight_ned/tokenizer_config.json',
 './flan_t5_lightweight_ned/tokenizer.json')

In [ ]:
def predict_answer(question):

    prompt = make_prompt(question)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=32,
        num_beams=4,
        early_stopping=True
    )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

In [ ]:
from tqdm import tqdm

preds = []

for q in tqdm(test_q):

    preds.append(predict_answer(q))

100%|██████████| 4000/4000 [09:04<00:00,  7.34it/s]


In [ ]:
for i in range(10):

    print("QUESTION :", test_q[i])
    print("PREDICTED:", preds[i])
    print("GOLD     :", test_a[i])
    print()

QUESTION : What man was a famous American author and also a steamboat pilot on the Mississippi River?
PREDICTED: William Henry Harrison
GOLD     : Mark Twain

QUESTION : How many Academy Awards has Jake Gyllenhaal been nominated for?
PREDICTED: 2
GOLD     : 1

QUESTION : Who is older, The Weeknd or Drake?
PREDICTED: Drake
GOLD     : Drake

QUESTION : How many children did Donald Trump have?
PREDICTED: 2
GOLD     : 5

QUESTION : Is the main hero in Final Fantasy IX named Kuja?
PREDICTED: True
GOLD     : False

QUESTION : Who performed at the Super Bowl XXIII halftime show?
PREDICTED: Bill Belichick
GOLD     : Elvis Presto

QUESTION : Did Free Guy come out in 2021?
PREDICTED: True
GOLD     : True

QUESTION : How many countries were in the Central Powers alliance in World War I?
PREDICTED: 3
GOLD     : 4

QUESTION : When was the first Donkey Kong arcade game released?
PREDICTED: 1988
GOLD     : 1981

QUESTION : Which movie, starring Al Jolson, is generally considered to be the first talki

In [ ]:
import re

def normalize(text):

    text = str(text).lower()

    text = re.sub(r"[^a-z0-9 ]", "", text)

    return " ".join(text.split())


def f1_score(pred, gold):

    pred_tokens = normalize(pred).split()
    gold_tokens = normalize(gold).split()

    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return 0

    common = set(pred_tokens) & set(gold_tokens)

    if len(common) == 0:
        return 0

    precision = len(common) / len(pred_tokens)
    recall = len(common) / len(gold_tokens)

    return 2 * precision * recall / (precision + recall)


hit1 = 0
f1 = 0

for p, g in zip(preds, test_a):

    if normalize(p) == normalize(g):
        hit1 += 1

    f1 += f1_score(p, g)

hit1 /= len(test_a)
f1 /= len(test_a)

print("="*50)
print("Flan-T5 + Lightweight NED")
print("="*50)
print("Hit@1    :", hit1)
print("Hit@5    :", hit1)
print("MRR       :", hit1)
print("F1        :", f1)
print("Accuracy  :", hit1)

Flan-T5 + Lightweight NED
Hit@1    : 0.21525
Hit@5    : 0.21525
MRR       : 0.21525
F1        : 0.2743348832540006
Accuracy  : 0.21525


In [ ]:
print(wikidata_search("Tom Cruise"))

Q37079


In [ ]:
import requests
import time

HEADERS = {
    "User-Agent": "FlanT5-Mintaka/1.0"
}

fact_cache = {}

USEFUL_PROPERTIES = {
    "instance of",
    "occupation",
    "country of citizenship",
    "spouse",
    "parent",
    "child",
    "educated at",
    "employer",
    "award received",
    "genre",
    "director",
    "producer",
    "author",
    "publisher",
    "performer",
    "screenwriter",
    "composer",
    "manufacturer",
    "country",
    "capital",
    "located in the administrative territorial entity",
    "continent",
    "part of",
    "date of birth",
    "date of death",
    "place of birth",
    "place of death",
    "main subject",
    "main character",
    "based on"
}

def retrieve_facts(entity_id, limit=20):

    if entity_id is None:
        return []

    if entity_id in fact_cache:
        return fact_cache[entity_id]

    query = f"""
    SELECT ?propertyLabel ?valueLabel
    WHERE {{
      wd:{entity_id} ?prop ?value .
      ?property wikibase:directClaim ?prop .

      SERVICE wikibase:label {{
        bd:serviceParam wikibase:language "en".
      }}
    }}
    """

    url = "https://query.wikidata.org/sparql"

    for attempt in range(3):

        try:

            response = requests.get(
                url,
                headers=HEADERS,
                params={
                    "query": query,
                    "format": "json"
                },
                timeout=30
            )

            if response.status_code != 200:
                time.sleep(2)
                continue

            data = response.json()

            facts = []

            for row in data["results"]["bindings"]:

                p = row["propertyLabel"]["value"]
                v = row["valueLabel"]["value"]

                if p in USEFUL_PROPERTIES:

                    facts.append((p, v))

            fact_cache[entity_id] = facts[:limit]

            return facts[:limit]

        except Exception:

            time.sleep(2)

    return []

In [ ]:
def choose_best_candidate(question, candidates):

    if len(candidates) == 0:
        return None

    q = question.lower()

    movie_words = [
        "movie", "film", "actor", "actress",
        "director", "star", "released", "release",
        "cinema"
    ]

    person_words = [
        "who", "born", "married", "wife",
        "husband", "children", "father",
        "mother", "author", "governor",
        "president", "singer"
    ]

    best = candidates[0]
    best_score = -100

    for cand in candidates:

        score = 0

        desc = cand.get("description", "").lower()
        label = cand.get("label", "").lower()

        # Film preference
        if any(w in q for w in movie_words):

            if "film" in desc:
                score += 10

            if "movie" in desc:
                score += 10

            if "video game" in desc:
                score -= 10

            if "ship" in desc:
                score -= 10

        # Person preference
        if any(w in q for w in person_words):

            if "human" in desc:
                score += 8

            if "actor" in desc:
                score += 10

            if "politician" in desc:
                score += 8

        if label in q:
            score += 3

        if score > best_score:
            best_score = score
            best = cand

    return best["id"]

In [ ]:
context_cache = {}

def build_context(question):

    if question in context_cache:
        return context_cache[question]

    entities = extract_entities(question)

    contexts = []

    for ent in entities:

        candidates = wikidata_search(ent)

        qid = choose_best_candidate(question, candidates)

        facts = retrieve_facts(qid)

        for p, v in facts:

            if isinstance(v, str) and v.startswith("http"):
                continue

            contexts.append(f"{p}: {v}")

    if len(contexts) == 0:
        context = "No knowledge available."
    else:
        context = " ; ".join(contexts)

    context_cache[question] = context

    return context

In [ ]:
print(build_context(train_q[1]))

producer: James Cameron ; producer: Jon Landau ; award received: Academy Award for Best Picture ; award received: Academy Award for Best Director ; award received: Academy Award for Best Original Song ; award received: Academy Award for Best Cinematography ; award received: Academy Award for Best Costume Design ; award received: Academy Award for Best Production Design ; award received: Academy Award for Best Film Editing ; award received: Academy Award for Best Visual Effects ; award received: Academy Award for Best Sound Editing ; award received: Japan Academy Prize for Outstanding Foreign Language Film ; award received: Golden Globe Award for Best Director ; award received: Academy Award for Best Sound ; award received: Golden Globe Award for Best Motion Picture – Drama ; award received: Saturn Award for Best Supporting Actress ; award received: Screen Actors Guild Award for Outstanding Performance by a Female Actor in a Supporting Role ; award received: Golden Globe Award for Best 

In [ ]:
for i in range(5):

    print("="*80)

    print("QUESTION:")
    print(train_q[i])

    print()

    context = build_context(train_q[i])

    print("CONTEXT:")
    print(context)

    print()

QUESTION:
What is the seventh tallest mountain in North America?

CONTEXT:
No knowledge available.

QUESTION:
Which actor was the star of Titanic and was born in Los Angeles, California?

CONTEXT:
No knowledge available.

QUESTION:
Which actor starred in Vanilla Sky and was married to Katie Holmes?

CONTEXT:
No knowledge available.

QUESTION:
What year was the first book of the A Song of Ice and Fire series published?

CONTEXT:
No knowledge available.

QUESTION:
Who is the youngest current US governor?

CONTEXT:
No knowledge available.



In [ ]:
qid = wikidata_search("Tom Cruise")

facts = retrieve_facts(qid)

print(facts[:10])

[('Bibliothèque nationale de France ID', '12199014b'), ('IdRef ID', '059677414'), ('NACSIS-CAT author ID', 'DA08938697'), ('IMDb ID', 'nm0000129'), ('NDL Authority ID', '00620541'), ('Commons category', 'Tom Cruise'), ('Libraries Australia ID', '35830645'), ('MusicBrainz artist ID', '05701f26-2083-41fa-9549-22ef7714b0c3'), ('unmarried partner', 'Penélope Cruz'), ('unmarried partner', 'Rebecca De Mornay')]


In [ ]:
qid = wikidata_search("Tom Cruise")

print(retrieve_facts(qid))

[('date of birth', '1962-07-03T00:00:00Z'), ('place of birth', 'Syracuse'), ('spouse', 'Nicole Kidman'), ('spouse', 'Katie Holmes'), ('spouse', 'Mimi Rogers'), ('country of citizenship', 'United States'), ('instance of', 'human'), ('child', 'Suri Cruise'), ('child', 'Isabella Jane Cruise'), ('child', 'Connor Cruise'), ('educated at', 'Glen Ridge High School'), ('educated at', 'Henry Munro Middle School'), ('educated at', 'St. Xavier High School'), ('occupation', 'screenwriter'), ('occupation', 'actor'), ('occupation', 'writer'), ('occupation', 'stunt performer'), ('occupation', 'musician'), ('occupation', 'aircraft pilot'), ('occupation', 'film director')]


In [ ]:
entities = extract_entities(q)

for e in entities:
    print(e)
    print(wikidata_search(e))

Titanic
Q25173
Los Angeles
Q65
California
Q99


In [ ]:
def make_prompt(question):

    context = build_context(question)

    prompt = f"""Answer the question using the given knowledge.

Question:
{question}

Knowledge:
{context}

Answer:
"""

    return prompt

In [ ]:
print(make_prompt(train_q[1]))

Answer the question using the given knowledge.

Question:
Which actor was the star of Titanic and was born in Los Angeles, California?

Knowledge:
producer: James Cameron ; producer: Jon Landau ; award received: Academy Award for Best Picture ; award received: Academy Award for Best Director ; award received: Academy Award for Best Original Song ; award received: Academy Award for Best Cinematography ; award received: Academy Award for Best Costume Design ; award received: Academy Award for Best Production Design ; award received: Academy Award for Best Film Editing ; award received: Academy Award for Best Visual Effects ; award received: Academy Award for Best Sound Editing ; award received: Japan Academy Prize for Outstanding Foreign Language Film ; award received: Golden Globe Award for Best Director ; award received: Academy Award for Best Sound ; award received: Golden Globe Award for Best Motion Picture – Drama ; award received: Saturn Award for Best Supporting Actress ; award re

In [ ]:
import json
from tqdm import tqdm

def create_inputs(questions, answers, save_prefix):

    inputs = []
    labels = []

    for i, (q, a) in enumerate(tqdm(zip(questions, answers),
                                    total=len(questions))):

        try:

            prompt = make_prompt(q)

        except Exception as e:

            print(f"Error at question {i}: {e}")

            prompt = f"""Answer the question.

Question:
{q}

Knowledge:
No knowledge available.

Answer:
"""

        inputs.append(prompt)
        labels.append(a)

        # Save every 500 examples
        if (i + 1) % 500 == 0:

            with open(f"{save_prefix}_inputs.json", "w") as f:
                json.dump(inputs, f)

            with open(f"{save_prefix}_labels.json", "w") as f:
                json.dump(labels, f)

            print(f"Saved {i+1} examples.")

    # Final save
    with open(f"{save_prefix}_inputs.json", "w") as f:
        json.dump(inputs, f)

    with open(f"{save_prefix}_labels.json", "w") as f:
        json.dump(labels, f)

    return inputs, labels

In [ ]:
train_inputs, train_labels = create_inputs(
    train_q,
    train_a,
    "train"
)

dev_inputs, dev_labels = create_inputs(
    dev_q,
    dev_a,
    "dev"
)

test_inputs, test_labels = create_inputs(
    test_q,
    test_a,
    "test"
)

  2%|▏         | 287/14000 [21:38<17:14:17,  4.53s/it]


KeyboardInterrupt: 

In [ ]:
fact_cache = {}
context_cache = {}

In [ ]:
print(build_context(train_q[0]))

part of: Americas ; instance of: continent ; instance of: subcontinent ; instance of: part of the world


In [ ]:
!nvidia-smi

Sat Jul  4 09:50:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   73C    P0             30W /   70W |    7071MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
print(train_ds.column_names)

['input_ids', 'attention_mask', 'labels']


In [ ]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

In [ ]:
batch = data_collator([train_ds[0], train_ds[1]])

batch = {k: v.to(device) for k, v in batch.items()}

model.train()

outputs = model(**batch)

print(outputs.loss)

tensor(2.8183, device='cuda:0', grad_fn=<NllLossBackward0>)
